# vidaudit — Qwen2.5-VL smoke test

End-to-end smoke run of the **canonical eval backend** ([Qwen/Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)) on Colab.

The point of this notebook is to verify that, on a fresh Colab T4, the full pipeline works without paid APIs:

1. Install vidaudit with the `qwen` extra (transformers + torch + accelerate).
2. Download a short test video and a hand-written descriptions file.
3. Run `vidaudit audit --backend qwen` and view the report.

**Runtime:** Set the runtime to T4 GPU (Runtime → Change runtime type → T4 GPU). The 3B checkpoint is ~7 GB in fp16, which fits comfortably. If you only have a smaller GPU, use the optional 4-bit cell at the bottom.

**Reproducibility (DD-14):** for the canonical eval, pin a commit SHA via `--qwen-revision <sha>`. This notebook leaves it unpinned for ease of running.

## 1. Install vidaudit + Qwen extras

Clone the repo and install in editable mode. The `[qwen]` extra pulls in `transformers>=4.49`, `torch`, and `accelerate`.

In [ ]:
!git clone https://github.com/shanemilek/vidaudit.git /content/vidaudit
%cd /content/vidaudit
!pip install -q -e '.[qwen]'
!python -m spacy download en_core_web_sm

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Grab a sample video + descriptions

We use a short Creative-Commons clip and write a small hand-crafted descriptions file with one accurate claim and one obvious hallucination so the audit has something to flag.

In [ ]:
import json
from pathlib import Path

DATA = Path("/content/sample_data")
DATA.mkdir(exist_ok=True)

video_url = "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/BigBuckBunny.mp4"
!wget -q -O {DATA}/clip.mp4 {video_url}
!ffprobe -v error -show_entries format=duration -of default=nw=1:nk=1 {DATA}/clip.mp4

descs = [
    {
        "timestamp_start": 10.0,
        "timestamp_end": 14.0,
        "description": "A large rabbit yawns near a tree in a green meadow.",
    },
    {
        "timestamp_start": 20.0,
        "timestamp_end": 24.0,
        "description": "A red sports car drives along a highway past the Eiffel Tower.",
    },
]
(DATA / "descs.json").write_text(json.dumps(descs, indent=2))
print("Wrote", DATA / "descs.json")

## 3. Run the audit

First time through, this downloads the ~7 GB Qwen2.5-VL-3B-Instruct checkpoint into the Colab disk. Subsequent runs are fast — frames and verification verdicts are both cached on disk (DD-11, DD-14).

In [ ]:
!vidaudit audit \
    --video /content/sample_data/clip.mp4 \
    --descriptions /content/sample_data/descs.json \
    --output /content/report.json \
    --backend qwen \
    --verbose

In [ ]:
import json

with open("/content/report.json") as f:
    report = json.load(f)
print(f"Backend:    {report['metadata']['backend']}")
print(f"Grounding:  {report['summary']['overall_grounding_score']:.2f}")
print(
    f"Flagged:    {report['summary']['descriptions_flagged']}/{report['summary']['total_descriptions']}"
)
for seg in report["segments"]:
    print(
        f"\n[{seg['timestamp_start']:.1f}-{seg['timestamp_end']:.1f}] {seg['verdict']}: {seg['description']}"
    )
    for c in seg["claims"]:
        print(f"  - ({c['verdict']}, conf={c['confidence']:.2f}) {c['text']}")

## 4. (Optional) 4-bit path for smaller GPUs

If you're not on a T4, run the cell below to install `bitsandbytes` and re-audit with `--qwen-4bit`. VRAM drops from ~7 GB to ~4 GB; accuracy delta on this verifier task is small but non-zero (a worth-comparing data point — see BACKLOG).

In [ ]:
!pip install -q bitsandbytes
!vidaudit audit \
    --video /content/sample_data/clip.mp4 \
    --descriptions /content/sample_data/descs.json \
    --output /content/report_4bit.json \
    --backend qwen \
    --qwen-4bit \
    --verbose